## 1. 派生変数（特徴量エンジニアリング）の生成に使用するクラス

<br>

#### ※ 派生変数の生成に関する代表的な方法

<div style="display:flex; justify-content:flex-start;">

<table style="
margin: 0 auto 0 0 !important;
width: auto;
border-collapse: collapse;
font-size: 14px;
">
<thead>
<tr>
<th style="padding:4px 10px;">方法 / クラス</th>
<th style="padding:4px 10px;">役割</th>
<th style="padding:4px 10px;">例</th>
</tr>
</thead>

<tbody>
<tr>
<td style="padding:4px 10px;"><code>PolynomialFeatures</code></td>
<td style="padding:4px 10px;">多項式・変数間の組み合わせを生成</td>
<td style="padding:4px 10px;"><code>X²</code>、<code>X³</code>、<code>X₁×X₂</code></td>
</tr>

<tr>
<td style="padding:4px 10px;"><code>FunctionTransformer</code></td>
<td style="padding:4px 10px;">任意の関数を適用して変換</td>
<td style="padding:4px 10px;"><code>log(X)</code>、<code>sqrt(X)</code></td>
</tr>
</tbody>
</table>

</div>

<br>

## 2. PolynomialFeatures クラス

`PolynomialFeatures` は、既存の入力変数 `X` を利用して、新しい変数（派生変数）を自動的に作成するクラスである。

<div style="color:#ff4d4f; font-weight:bold; margin:12px 0;">
「既存の変数の組み合わせを利用して、問題をよりよく説明できる新しい変数（特徴量）を作る」
</div>

<div style="color:#ff4d4f; margin-bottom:16px;">
派生変数は、既存の変数の組み合わせ・変換を利用して新しい情報を作り、モデルがデータの関係をよりよく学習できるようにする。<br>
その結果、モデルの性能が向上する場合もある。
</div>

`PolynomialFeatures` は、既存の数値型変数の**べき乗**や**変数間の積**を自動的に作成する sklearn のクラスである。

<div style="
background: repeating-linear-gradient(
    to bottom,
    #f4f5f6 0px,
    #f4f5f6 27px,
    #e7e9eb 27px,
    #e7e9eb 54px
);
padding: 0 8px;
margin: 0;
font-family: Consolas, 'Yu Gothic', monospace;
font-size: 14px;
line-height: 27px;
color: #111111;
white-space: nowrap;
overflow-x: auto;
">&nbsp;<br>【なぜこのような派生変数を作るのか？】<br><br>&#45; 線形回帰は基本的に <span style="color:#e67e22;">y = b₀ + b₁X</span> のような直線関係を学習する。<br>&#45; しかし、実際のデータが <span style="color:#e67e22;">X</span> の増加に伴って、最初は急激に増加し、その後は増加幅が小さくなる形であれば、<br>&nbsp;&nbsp;&nbsp;単純な直線では表現しにくい場合がある。<br>→ したがって、<span style="color:#e67e22;">X²</span> のような派生変数が必要になる。<span style="color:#e67e22;">X²、X³</span> のような変数を追加すると、線形回帰モデルでも曲線形の関係を表現できるようになる。<br><br>&#45; x1、x2 の2つの独立変数だけを使用する場合<span style="color:#ff4d4f;">よりも</span>、(x1 × x2) のような派生変数を追加することで、よりうまく表現できる。<br>&nbsp;</div>

<br>

`PolynomialFeatures` クラス生成時の `degree` パラメータを使用して、高次項の最大次数を指定できる。

- `degree=3` → 3次項まで生成

変数が少なすぎても問題だが、多すぎると訓練データに対してのみ高い性能を示す  
<span style="color:#ff4d4f; font-weight:bold;">「過学習」</span> が発生する。

→ テストデータでは低いスコアになる。

<br>

#### 1) 派生変数を追加しない場合

In [4]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df = pd.DataFrame({
    "x": [0, 1, 2, 3, 4, 5, 6],
    "y": [2, 4, 9, 19, 37, 64, 102]
})

X = df[["x"]]
y = df["y"]

model2 = LinearRegression()
model2.fit(X, y)

y_pred = model2.predict(X)

print("予測値:", y_pred.round(2))
print("R²:", round(r2_score(y, y_pred), 4))
print("MSE:", round(mean_squared_error(y, y_pred), 4))

予測値: [-14.14   1.86  17.86  33.86  49.86  65.86  81.86]
R²: 0.8629
MSE: 162.6939


<br>

<div style="
background: repeating-linear-gradient(
    to bottom,
    #e7e9eb 0px,
    #e7e9eb 27px,
    #f4f5f6 27px,
    #f4f5f6 54px
);
padding: 0 8px;
margin: 0;
font-family: Consolas, 'Yu Gothic', monospace;
font-size: 14px;
line-height: 27px;
color: #111111;
white-space: nowrap;
overflow-x: auto;
">&nbsp;<br><span style="color:#e67e22;">■ R²</span> → <strong>R-squared</strong>：決定係数<br>
- <strong>モデルの説明力を表す指標</strong>。R²は、モデルの予測誤差を「平均値で予測した場合の誤差」と比較した指標である。<br>
- <span style="color:#e67e22;"><strong>1 に近いほど</strong></span>、データの変動をよりよく説明できる。<br><br>

<span style="color:#e67e22;">■ MSE</span> → <strong>Mean Squared Error</strong>：平均二乗誤差<br>
- <strong>実際値と予測値の差を二乗し、その平均を求めた値</strong>である。大きな誤差ほど強く反映される。<br>
- <span style="color:#e67e22;"><strong>値が小さいほど</strong></span>、予測誤差が小さく性能が良い。<br>
- 単位は<strong>元の値の二乗単位</strong>になる。
</div>

<br>

#### 2) `degree` パラメータ：2

In [5]:
poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_poly2 = poly2.fit_transform(X)

model2 = LinearRegression()
model2.fit(X_poly2, y)
# y = a*x² + b*x + c

y_pred2 = model2.predict(X_poly2)

print("degree=2 特徴量:", poly2.get_feature_names_out(["x"]))
print("予測値:", y_pred2.round(2))
print("R²:", round(r2_score(y, y_pred2), 4))
print("MSE:", round(mean_squared_error(y, y_pred2), 4))

degree=2 特徴量: ['x' 'x^2']
予測値: [  4.07   1.86   6.93  19.29  38.93  65.86 100.07]
R²: 0.9971
MSE: 3.449


<br>

#### 3) `degree` パラメータ：3

In [6]:
poly3 = PolynomialFeatures(degree=3, include_bias=False)
X_poly3 = poly3.fit_transform(X)

model3 = LinearRegression()
model3.fit(X_poly3, y)
# y = a·x³ + b·x² + c·x + d

y_pred3 = model3.predict(X_poly3)

print("degree=3 特徴量:", poly3.get_feature_names_out(["x"]))
print("予測値:", y_pred3.round(2))
print("R²:", round(r2_score(y, y_pred3), 4))
print("MSE:", round(mean_squared_error(y, y_pred3), 4))

degree=3 特徴量: ['x' 'x^2' 'x^3']
予測値: [  2.07   3.86   8.93  19.29  36.93  63.86 102.07]
R²: 1.0
MSE: 0.0204


<br>

【解釈】

学習に使用される特徴量の増加  
`['x' 'x^2']` → `['x' 'x^2' 'x^3']`

→ R² 増加  
→ MSE 減少  

モデルの性能が向上したと考えられる。